In [7]:
!unzip -q "archive (1).zip" -d dataset

In [8]:
import torch
import torch.nn as nn
import torchvision
import torch.optim as optim
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, confusion_matrix


In [9]:
# DataSet And Dataloader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms # help transforming photos to CNN
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


In [10]:
trainset = ImageFolder(root="dataset/train",transform=transform)
valset = ImageFolder(root="dataset/val",transform=transform)

In [11]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(valset,batch_size=64)

In [12]:

class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()
    self.conv_layers = nn.Sequential(
        nn.Conv2d(3,32,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
)
    self.fc_layers = nn.Sequential(
        nn.Linear(16*16*128,256),
        nn.ReLU(),
        nn.Linear(256,2)
    )
  def forward(self,x):
    x=self.conv_layers(x)
    x= x.view(x.size(0),-1) #flatning
    x=self.fc_layers(x)
    return x

In [13]:
model=CNN()

In [14]:
criterian = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [15]:
#train CNN

train_loss =[]
val_loss = []
epochs = 10
for epoch in range(epochs):
    model.train()  # Model ko training mode me set kiya
    epoch_running_loss = 0.0


    for images,labels in trainloader:
      optimizer.zero_grad()
      outputs = model.forward(images) # take model prediction
      loss = criterian(outputs,labels) #loss calculate
      loss.backward() #know the wrong loss
      optimizer.step() #update params

      epoch_running_loss += loss.item() #loss track

    epoch_training_loss= epoch_running_loss/len(trainloader)
    train_loss.append(epoch_training_loss)

    #validation
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
      for images,labels in testloader:
        outputs = model.forward(images)
        loss = criterian(outputs,labels)
        running_val_loss += loss.item()

    epoch_val_loss = running_val_loss/len(testloader)
    val_loss.append(epoch_val_loss)
    print(f"{epoch+1}/{epochs}==> training loss{epoch_training_loss} & validation loss {epoch_val_loss}")

1/10==> training loss0.8942182540893555 & validation loss 0.723958283662796
2/10==> training loss0.659208106994629 & validation loss 0.48259709775447845
3/10==> training loss0.6661473631858825 & validation loss 0.5700477659702301
4/10==> training loss0.641244113445282 & validation loss 0.5434118360280991
5/10==> training loss0.6500027894973754 & validation loss 0.5100863128900528
6/10==> training loss0.6311401128768921 & validation loss 0.5682743638753891
7/10==> training loss0.6137813448905944 & validation loss 0.4935372918844223
8/10==> training loss0.6112998843193054 & validation loss 0.5428907722234726
9/10==> training loss0.5766010284423828 & validation loss 0.5357352644205093
10/10==> training loss0.597498208284378 & validation loss 0.604301780462265


In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# 1. Lists to store all actual values and predictions
all_preds = []
all_labels = []

# 2. Loop through validation/test set
with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        # Append to CPU lists
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

accuracy  = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
recall    = recall_score(all_labels, all_preds, average='macro', zero_division=0)
cm        = confusion_matrix(all_labels, all_preds)
print(f"Accuracy         : {accuracy:.4f}")
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro)   : {recall:.4f}")
print("Confusion Matrix :\n", cm)

Accuracy         : 0.6429
Precision (Macro): 0.5663
Recall (Macro)   : 0.5389
Confusion Matrix :
 [[ 5 19]
 [ 6 40]]
